In [1]:
import torch

In [18]:
p = torch.tensor([[1,2],[3,4]])
print(p.shape[0])
q = p.unsqueeze(0).repeat([11] + [1]*2)
print(q.shape)
q = p
print(len(p.shape))

if p.shape == q.shape:
    print('glhf')

2
torch.Size([11, 2, 2])
2
glhf


In [10]:
import torch

from torchrl.envs import (
    check_env_specs,
    ExplorationType,
    PettingZooEnv,
    RewardSum,
    set_exploration_type,
    TransformedEnv,
    VmasEnv,
)
from torchrl.modules import (
    AdditiveGaussianModule,
    MultiAgentMLP,
    ProbabilisticActor,
    TanhDelta,
)
from tensordict.nn import TensorDictModule, TensorDictSequential

from tensordict import TensorDict

from torchrl.envs.utils import RandomPolicy

from metamaterial_envs import crawler_v0

device = (
    torch.device(0)
    if torch.cuda.is_available() and not is_fork
    else torch.device("cpu")
)

In [11]:
vmas_env = VmasEnv(
        scenario="flocking",
        num_envs=10,
        continuous_actions=True,
        max_steps=100,
        device=device,
        seed=0,
        # Scenario specific
)
meta_env = crawler_v0.torch_env(num_envs=10, num_particles=15)

In [12]:
# vmas_actor = RandomPolicy(action_spec=vmas_env.action_spec)
# action = vmas_actor(TensorDict({}, batch_size=vmas_env.action_spec.shape))
# print(action)
# print(vmas_env.step(action))

# meta_actor = RandomPolicy(action_spec=meta_env.action_spec)
# action = meta_actor(TensorDict({}, batch_size=meta_env.action_spec.shape))
# print(meta_env.step(action))

In [14]:
env = meta_env
share_parameters_policy = True
total_frames = 100_000

policy_net = MultiAgentMLP(
    n_agent_inputs=env.observation_spec["agents", "observation"].shape[
        -1
    ],  # n_obs_per_agent
    n_agent_outputs=env.full_action_spec["agents", "action"].shape[
        -1
    ],  # n_actions_per_agents
    n_agents=env.num_agents,  # Number of agents in the group
    centralised=False,  # the policies are decentralised (i.e., each agent will act from its local observation)
    share_params=share_parameters_policy,
    device=device,
    depth=2,
    num_cells=256,
    activation_class=torch.nn.Tanh,
)
# Wrap the neural network in a :class:`~tensordict.nn.TensorDictModule`.
# This is simply a module that will read the ``in_keys`` from a tensordict, feed them to the
# neural networks, and write the
# outputs in-place at the ``out_keys``.
policy_module = TensorDictModule(
    policy_net,
    in_keys=[("agents", "observation")],
    out_keys=[("agents", "param")],
)  # We just name the input and output that the network will read and write to the input tensordict
policy = ProbabilisticActor(
    module=policy_module,
    spec=env.full_action_spec["agents", "action"],
    in_keys=[("agents", "param")],
    out_keys=[("agents", "action")],
    distribution_class=TanhDelta,
    distribution_kwargs={
        "low": env.full_action_spec_unbatched["agents", "action"].space.low,
        "high": env.full_action_spec_unbatched["agents", "action"].space.high,
    },
    return_log_prob=False,
)

exploration_policy = TensorDictSequential(
    policy,
    AdditiveGaussianModule(
        spec=policy.spec,
        annealing_num_steps=total_frames // 2,  # Number of frames after which sigma is sigma_end
        action_key=("agents", "action"),
        sigma_init=0.9,  # Initial value of the sigma
        sigma_end=0.1,  # Final value of the sigma
    ),
)

In [54]:
batch_size = (10, 10)
n_agents = env.num_agents
observation_size = env.observation_spec['agents', 'observation'].shape[-1]

x = torch.linspace(-torch.pi, torch.pi, batch_size[0])
y = torch.stack(torch.meshgrid(x, x)).transpose(0, 2).unsqueeze(2).repeat(1, 1, n_agents, 1)

with torch.no_grad():
    td = TensorDict(
            {
                "agents": TensorDict(
                    {
                        "observation": y
                    },
                    batch_size = torch.Size([*batch_size, n_agents]),
                    device = env.device
                )
            },
            batch_size = torch.Size([*batch_size]),
            device = env.device
        )
    hm = policy(td)['agents', 'action'].squeeze(-1).numpy()
hm

array([[[-0.63785243, -0.63785243, -0.63785243, ..., -0.63785243,
         -0.63785243, -0.63785243],
        [-0.46257135, -0.46257135, -0.46257135, ..., -0.46257135,
         -0.46257135, -0.46257135],
        [-0.2715202 , -0.2715202 , -0.2715202 , ..., -0.2715202 ,
         -0.2715202 , -0.2715202 ],
        ...,
        [ 0.18260548,  0.18260548,  0.18260548, ...,  0.18260548,
          0.18260548,  0.18260548],
        [ 0.2857116 ,  0.2857116 ,  0.2857116 , ...,  0.2857116 ,
          0.2857116 ,  0.2857116 ],
        [ 0.3982088 ,  0.3982088 ,  0.3982088 , ...,  0.3982088 ,
          0.3982088 ,  0.3982088 ]],

       [[-0.46520793, -0.46520793, -0.46520793, ..., -0.46520793,
         -0.46520793, -0.46520793],
        [-0.39671826, -0.39671826, -0.39671826, ..., -0.39671826,
         -0.39671826, -0.39671826],
        [-0.18038337, -0.18038337, -0.18038337, ..., -0.18038337,
         -0.18038337, -0.18038337],
        ...,
        [ 0.44127053,  0.44127053,  0.44127053, ...,  

In [59]:
env.rollout(9)["log_info", "trajectory"].numpy().shape

(10, 9, 15, 2)